# Datamine QUANTILE Process Example

This notebook demonstrates how to configure and run the **`quantile`** process wrapper in `dmstudio`.

## Process Description

## QUANTILE

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Collectively quartiles, deciles, percentiles and other values obtained by equal subdivision of data are called quantiles. Quantiles give information about the shape of a distribution; in particular whether a distribution is skewed or not. Quantiles can be used for comparing two distributions.

The **QUANTILE** process carries out two types of analysis on a set of sample data:

* Quantile Point Analysis: a table showing the grades corresponding to user defined percentile values.
* Quantile Group Analysis: a set of tables showing the statistics of the grades lying between consecutive pairs of user defined percentile values

The first stage of the **QUANTILE** process is to apply the cutoff grade as specified by parameter **CUTOFF**. Any sample below the cutoff grade is removed from the analysis. Using the default cutoff grade of 0 in effect means that no cutoff is used.

The second stage is to apply a topcut grade. If the parameter **TOPCUT** has been set to 1, then any sample which is greater than parameter **TOPGRADE** will be replaced by a sample of **TOPGRADE**. If **TOPCUT** is set to 0, then the topcut will not be used.

The sample file is sorted in ascending order and divided into equal numbers of samples as defined by parameter **QUANTIL1**. For example, if there are 120 samples in total and **QUANTIL1** =10, then each subdivision or bin will include 12 samples. If the total number of samples does not divide equally by **QUANTIL1** , then some bins will contain one more sample than others.

If **QUANTIL1** =8 then the first bin will contain the lowest grade 12.5% of samples, bin 2 will contain the samples between 12.5% and 25%, and so on, with the top bin, bin 8, contain the highest 12.5% of samples. The split using the **QUANTIL1** parameter is called the primary subdivision.

The top bin can be further divided, as controlled by parameter **QUANTIL2**. For example if **QUANTIL2** =5, then 5 additional bins from 87.5% - 90%, 90% - 92.5%, etc will be calculated. This is the secondary subdivision. If you do not want a secondary subdivision then QUANTIL2 should be set to zero.

An example of a Quantile Point table, file **QUANT_PT** , is shown below. This has been created using **QUANTIL1** =10 and **QUANTILE2** =4. A set of results is shown for each zone if the KEY field has been selected:

![](../Images/QUANTIILE1.jpg)

The Group Analysis tables show statistics between each pair of Percentiles; 0% - 10%, 10% - 20%, and so on. The following statistics are calculated for each bin:

* the number of samples

* minimum grade

* maximum grade

* mean grade

* metal content (the sum of the individual grade values)

* % metal in bin (the metal content as a percentage of total metal)

Two Group Analysis output files can be created. The RESULTS file includes both the primary and secondary divisions and the PRIMARY file includes just the primary divisions. The Results table can be saved to a system text file if file PRINT is specified. The table will also be displayed in the Command Window. An example of the RESULTS file is shown below:

[![](../Images/QUANTILE2.jpg)](<javascript:void\(0\);>)

If a **WEIGHT** field has been specified then the mean grade is a weighted mean and the metal content is the sum of weight*grade. Also the values in the **NSAMPLES** (Number of Samples) column are not necessarily equal because the weights, not the number of samples, are equally distributed between the quantiles. The **WEIGHT** field applies to both the Point and Group analysis tables.

If a **KEY** field has been specified, then the quantile analysis is done separately for each value of the **KEY** field.

### Input Files:

* **in** (Undefined):
  Input sample file
  Required=Yes

### Output Files:

* **quant_pt** (Table File):
  Output file containing the * **VALUE** value for each Quantile Point defined by
  parameters **QUANTIL1** and **QUANTIL2**. Although optional one of the two files
  **QUANT_PT** or **RESULTS** must be selected.
  Required=No

* **results** (Table File):
  Output file containing quantile group information for primary and secondary
  subdivisions. Although optional one of the two files **QUANT_PT** or **RESULTS** must be
  selected.
  Required=Yes

* **primary** (Table File):
  Output file containing quantile group information for the primary subdivision only.
  Required=No

* **print** (Table File):
  System print file, containing quantile group information. This is a copy of the
  contents of the **RESULTS** file, but to a system file. The extension .pri will be added
  automatically to the file name.
  Required=No

### Fields:

* **value** (Numeric : IN):
  Name of the field containing the grade to be analysed.
  Default=Undefined
  Required=Yes

* **key** (Numeric : IN):
  Key field for grouping the data. A separate quantile analysis is carried out for each
  unique value of the key field.
  Default=Undefined
  Required=No

* **weight** (Numeric : IN):
  Field containing the weight used when calculating quantile means and accumulating the
  grade values. For example if the input data is a desurveyed data file then **LENGTH**
  could be used. If no field is specified then all samples have an equal weight.
  Default=Undefined
  Required=No

### Parameters:

* **quantil1**:
  The primary quantile. The number of primary subdivisions or bins for grouping the
  samples. For example setting **QUANTIL1** =10 will divide the samples into deciles;
  **QUANTIL1** =4 will give quartiles.
  Range=2,+
  Values=Undefined
  Default=10
  Required=No

* **quantil2**:
  The secondary quantile. The top bin of the primary subdivision can be further split
  into equal groupings. For example if **QUANTIL1** =10 and **QUANTIL2** =4, then the top
  10% will be split into 4 equal groups of 2.5%. If set to 0 then the top bin is not
  resplit.
  Range=0,+
  Values=Undefined
  Default=0
  Required=No

* **cutoff**:
  Cutoff grade. Only samples greater than or equal to the cutoff grade are selected for
  analysis.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **topcut**:
  Flag to specify whether or not a topcut grade should be applied: 0 = No topcut applied.
  1 = Topcut applied at grade defined in **TOPGRADE**
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **topgrade**:
  Grade to be applied as a topcut, if **TOPCUT** is set to 1. Any value greater than
  **TOPGRADE** will be reset to equal to **TOPGRADE**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ndp**:
  Maximum number of decimal places for field **VALUE** in output file **QUANT_PT**.
  Range=0,6
  Values=0,1,2,3,4,5,6
  Default=2
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('quantile')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute quantile
print("Running quantile...")
dm_cmd.quantile(
    in_i='t_assays',  # required
    quant_pt_o='t_quantile_out',  # required
    results_o=['t_quantile_out'],  # required
    print_o='t_quantile_out',  # required
    value_f='optional',  # required
    # primary_o='t_quantile_out',  # optional
    # key_f=['BHID'],  # optional
    # weight_f='optional',  # optional
    # quantils_f=['optional'],  # optional
    # cutoff_p=0,  # optional
    # topcut_p=0,  # optional
    # topgrade_p='optional',  # optional
    # ndp_p=2,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("quantile execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_quantile_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")